A notebook for prompting of SOTA LLMs.

## Data Loading

In [45]:

import json
import re

import os
from pathlib import Path

with open("not_to_upload.txt", "r") as f:
    API_KEY = f.read().split("\n")
    
with open("system_prompt.md", "r") as f:
    system_prompt = f.read()
    
with open("definitions.txt", "r") as f:
    definitions_str = f.read()

definitions_dict = [{"label": definition.split(" --- ")[0], "definition":definition.split(" --- ")[1]} for definition in definitions_str.split("\n")]

def create_string(def_dict):
    definitions_string = ""
    for ent in def_dict:
        definitions_string = definitions_string + ent["label"] + " - " + ent["definition"] + "\n"
        
    return definitions_string

system_prompt_withdefinitions = re.sub(r"\[TAXONOMY - DEFINITIONS - RULES\]", create_string(definitions_dict), system_prompt)

In [66]:
DATA_DIR = "RESULTS/annots_v2"

In [70]:
files = os.listdir(DATA_DIR)

sentences = {}

for file in files:
    file_dir = Path.joinpath(Path(DATA_DIR),Path(file))
    print(file_dir)

    with open(file_dir, "r") as f:
        data_json = json.load(f)
        
    for i in range(len(data_json)):
        paper_id = data_json[i]["data"]["paper_id"]
        sentence_id = data_json[i]["data"]["sentence_id"]
        
        compound = f"{paper_id}-{sentence_id}"
        # print(compound)
        sentences[f"{compound}"]= data_json[i]["data"]["sentence"]

print(len(sentences))



RESULTS/annots_v2/mmp_50_MatExp_MeaDev_PhyPhe_Qua.json
RESULTS/annots_v2/mmp_50_EneSou_MetPhe_NatDis_NatPhe.json
RESULTS/annots_v2/mmp_50_Ass_Pol.json
RESULTS/annots_v2/mmp_50_PhyArt_Org_Per_Sys.json
RESULTS/annots_v2/mmp_50_Loc_GeoFea_BodOfWat_TimPer_Sat.json
RESULTS/annots_v2/mmp_50_BodPar_Che_Dis_Org_Eco.json
RESULTS/annots_v2/mmp_50_Met_FieOfStu_IntArt.json
192


## Gemini

In [ ]:
# !pip install google-genai
from google import genai
from google.genai import types

In [ ]:

# 1. Setup
client = genai.Client(api_key=API_KEY[0])

system_instruction = system_prompt_withdefinitions

# 3. User Input
user_input = "The experimental results showed that the pH levels dropped by 50% in the Danube River."

# 4. Call API with JSON enforcement
response = client.models.generate_content(
    model="gemini-3-pro-preview",
    contents=user_input,
    config=types.GenerateContentConfig(
        system_instruction=system_instruction,
        response_mime_type="application/json", # Forces valid JSON output
        temperature=0.1 # Low temp is better for extraction tasks
    )
)

# 5. Process Result
try:
    data = json.loads(response.text)
    
    # Example logic to find spans (naive string match)
    for item in data:
        entity = item['entity_text']
        start_index = user_input.find(entity)
        if start_index != -1:
            end_index = start_index + len(entity)
            print(f"Found '{entity}' ({item['category']}) at [{start_index}:{end_index}]")
            print(f"Reason: {item['reasoning']}\n")
        else:
            print(f"Warning: Could not locate '{entity}' exactly in source.")
            
except json.JSONDecodeError:
    print("Model failed to generate valid JSON.")
    print(response.text)

## ChatGPT

In [ ]:
# !pip install openai
from openai import OpenAI

In [ ]:
# 1. Setup
client = OpenAI(api_key=API_KEY[1])

system_instruction = system_prompt_withdefinitions

# 2. User Input
user_input = "The experimental results showed that the pH levels dropped by 50% in the Danube River."

# 3. ChatGPT API Call with JSON enforcement
response = client.chat.completions.create(
    model="gpt-4.1",
    messages=[
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": user_input}
    ],
    response_format={"type": "json_object"},  # Force valid JSON
    temperature=0.1
)

# 4. Extract JSON text
response_json = response.choices[0].message.content

# 5. Process Result
try:
    data = json.loads(response_json)

    # Example logic to find spans (naive string match)
    for item in data:
        entity = item["entity_text"]
        start_index = user_input.find(entity)
        if start_index != -1:
            end_index = start_index + len(entity)
            print(f"Found '{entity}' ({item['category']}) at [{start_index}:{end_index}]")
            print(f"Reason: {item['reasoning']}\n")
        else:
            print(f"Warning: Could not locate '{entity}' exactly in source.")

except json.JSONDecodeError:
    print("Model failed to generate valid JSON.")
    print("Raw output:\n", response_json)


## DeepSeek

In [31]:
from openai import OpenAI

In [35]:

# 1. Setup
client = OpenAI(
    api_key=API_KEY[2],
    base_url="https://api.deepseek.com"
)

system_instruction = system_prompt_withdefinitions

# 2. User Input
user_input = "The experimental results showed that the pH levels dropped by 50% in the Danube River."

# 3. Call API with JSON enforcement
response = client.chat.completions.create(
    model="deepseek-chat",  # or "deepseek-coder" for coding tasks
    messages=[
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": user_input}
    ],
    response_format={"type": "json_object"},  # Forces valid JSON output
    temperature=0.1,  # Low temp is better for extraction tasks
    stream=False
)

# 4. Process Result
try:
    # Extract the JSON response
    response_text = response.choices[0].message.content
    data = json.loads(response_text)
    
    # Example logic to find spans (naive string match)
    for item in data:
        entity = item['entity_text']
        start_index = user_input.find(entity)
        if start_index != -1:
            end_index = start_index + len(entity)
            print(f"Found '{entity}' ({item['category']}) at [{start_index}:{end_index}]")
            print(f"Reason: {item['reasoning']}\n")
        else:
            print(f"Warning: Could not locate '{entity}' exactly in source.")
            
except json.JSONDecodeError:
    print("Model failed to generate valid JSON.")
    print(response_text)
except Exception as e:
    print(f"Error: {e}")

APIStatusError: Error code: 402 - {'error': {'message': 'Insufficient Balance', 'type': 'unknown_error', 'param': None, 'code': 'invalid_request_error'}}